In [1]:
# WD0806-661 B batman detectability notebook, v16
# Keep wd0806_batman_detectability_v16.py in the same folder as this notebook.

%matplotlib inline

import importlib
import time

import matplotlib.pyplot as plt
import numpy as np

import wd0806_batman_detectability_v16 as wd0806

wd0806 = importlib.reload(wd0806)
print("Loaded:", wd0806.__file__)


Loaded: C:\Users\janse\PSLU Python code\P3\P3 detectability\FATSO_alpha\Finale2\wd0806_batman_detectability_v16.py


In [2]:
# Install with `pip install batman-package` if this check fails.
try:
    wd0806.require_batman()
    print("batman import OK")
except ImportError as exc:
    print(exc)


batman import OK


In [3]:
# Select one editable noise model.
noise_model = wd0806.jwst_like_noise()
# noise_model = wd0806.placeholder_direct_imaging_noise()
# noise_model = wd0806.pessimistic_direct_imaging_noise()

cfg = wd0806.make_wd0806_config(seed=7, noise=noise_model)
cfg.n_transits = 10
cfg.local_window_days = None
cfg.snr_threshold = 7.0
cfg.logk_threshold = np.log(10.0)

# Optional direct edits.
# cfg.noise.cadence_min = 10.0
# cfg.noise.white_ppm = 2500.0
# cfg.noise.red_ppm = 500.0
# cfg.noise.red_timescale_hr = 2.0
# cfg.body_grid.radius_re_min = 0.2
# cfg.body_grid.radius_re_max = 2.5
# cfg.body_grid.a_rprimary_min = 2.5
# cfg.body_grid.a_rprimary_max = 35.0

for name, table in wd0806.parameter_input_dataframes(cfg).items():
    print("\n" + name)
    display(table)



primary


,name,mass_mj,radius_rj,limb_dark,u
0,WD 0806-661 B,7.0,1.0,uniform,()



body_grid


,radius_re_min,radius_re_max,a_rprimary_min,a_rprimary_max,impact_b_max,ecc_min,ecc_max,omega_deg
0,0.2,2.5,2.5,35.0,0.85,0.0,0.05,90.0



noise


,instrument,cadence_min,white_ppm,red_ppm,red_timescale_hr,duty_cycle
0,jwst_like_comparison_not_etc,5.0,1000.0,300.0,2.0,1.0



experiment


,n_transits,local_window_days,rng_seed,snr_threshold,logk_threshold,supersample_factor,batman_max_err_ppm
0,10,None,7,7.0,2.302585,5,1.0


In [ ]:
# Available run sizes: smoke, quick, standard, long.
t0 = time.time()
results = wd0806.run_wd0806_batman_pipeline(
    cfg=cfg,
    run_size="long",
    seed=cfg.rng_seed,
    progress=True,
)
print(f"Grid runtime: {time.time() - t0:.2f} seconds")


body grid 1/30, 1/30: R=0.2 Re, a=2.5 Rprimary
body grid 1/30, 2/30: R=0.2 Re, a=2.74 Rprimary
body grid 1/30, 3/30: R=0.2 Re, a=3 Rprimary
body grid 1/30, 4/30: R=0.2 Re, a=3.28 Rprimary
body grid 1/30, 5/30: R=0.2 Re, a=3.6 Rprimary
body grid 1/30, 6/30: R=0.2 Re, a=3.94 Rprimary
body grid 1/30, 7/30: R=0.2 Re, a=4.32 Rprimary
body grid 1/30, 8/30: R=0.2 Re, a=4.73 Rprimary
body grid 1/30, 9/30: R=0.2 Re, a=5.18 Rprimary
body grid 1/30, 10/30: R=0.2 Re, a=5.67 Rprimary
body grid 1/30, 11/30: R=0.2 Re, a=6.21 Rprimary
body grid 1/30, 12/30: R=0.2 Re, a=6.8 Rprimary
body grid 1/30, 13/30: R=0.2 Re, a=7.45 Rprimary
body grid 1/30, 14/30: R=0.2 Re, a=8.16 Rprimary
body grid 1/30, 15/30: R=0.2 Re, a=8.94 Rprimary
body grid 1/30, 16/30: R=0.2 Re, a=9.79 Rprimary
body grid 1/30, 17/30: R=0.2 Re, a=10.7 Rprimary
body grid 1/30, 18/30: R=0.2 Re, a=11.7 Rprimary
body grid 1/30, 19/30: R=0.2 Re, a=12.9 Rprimary
body grid 1/30, 20/30: R=0.2 Re, a=14.1 Rprimary
body grid 1/30, 21/30: R=0.2 Re, a=

In [ ]:
display(results["metrics"])
print(results["report"])


In [ ]:
# Set True to inspect the complete grid tables.
SHOW_GRID_TABLES = False

if SHOW_GRID_TABLES:
    print("Body recovery grid")
    display(results["body_grid"])
    print("No-object false-positive grid")
    display(results["noise_grid"])


In [ ]:
fig, ax = wd0806.plot_recovery_heatmap(results)
plt.show()

fig, ax = wd0806.plot_uncertainty_heatmap(results)
plt.show()

fig, ax = wd0806.plot_false_positive_heatmap(results)
plt.show()


In [ ]:
body_trials = results["body_trials"]
noise_trials = results["noise_trials"]

k_rec = int(body_trials["detected"].sum())
n_body = len(body_trials)
k_fp = int(noise_trials["detected"].sum())
n_noise = len(noise_trials)

TPR = k_rec / n_body if n_body else np.nan
FPR = k_fp / n_noise if n_noise else np.nan
K_grid_proxy = TPR / FPR if FPR > 0 else np.inf
lnK_grid_proxy = np.log(K_grid_proxy) if np.isfinite(K_grid_proxy) else np.inf

print("Whole-grid trigger summary")
print("==========================")
print(f"Injected batman transits recovered: {k_rec} / {n_body} = {TPR:.4f} ({100 * TPR:.2f}%)")
print(f"No-object false positives: {k_fp} / {n_noise} = {FPR:.4f} ({100 * FPR:.2f}%)")
print(f"K_grid_proxy = TPR/FPR = {K_grid_proxy:.6g}")
print(f"ln(K_grid_proxy) = {lnK_grid_proxy:.6g}")


In [ ]:
paths = wd0806.save_grid_results(
    results,
    output_dir="result_wd0806_v16_long",
)

for key, path in paths.items():
    print(f"{key}: {path}")


## Optional nested-sampling validation

The nested runs are diagnostic comparisons between a flat no-object model, \(M0\), and a batman transit model, \(M1\). The main detectability result remains the injection--recovery grid together with its no-object false-positive grid.

The module retains UltraNest's default HDF5 storage for general use, but the calls below select `storage_backend="csv"`. This avoids the Windows HDF5 file-lock error that can occur when a notebook reruns an existing output directory. Best, worst, and intermediate cases are also run independently and written to separate directories.


In [ ]:
# Reload the edited module, then prepare the three deterministic validation bodies.
wd0806 = importlib.reload(wd0806)

validation_bodies = wd0806.nested_validation_case_bodies(
    cfg,
    include_intermediate=True,
)
print("Available nested-validation cases:", list(validation_bodies))


def nested_config(max_ncalls):
    """Shared settings; pass None for an uncapped run."""
    return wd0806.NestedConfig(
        min_live_points=200,
        dlogz=0.5,
        max_ncalls=max_ncalls,
        best_case_max_ncalls=None,
        resume="overwrite",
        use_stepsampler=True,
        stepsampler_nsteps=50,
        stepsampler_adaptive_nsteps="move-distance",
        stepsampler_region_filter=True,
        frac_remain=0.5,
        storage_backend="csv",
    )


In [ ]:
RUN_NESTED_BEST = True
nested_best = None

if RUN_NESTED_BEST:
    best_ns_cfg = nested_config(max_ncalls=100_000)

    # Continue from saved likelihood evaluations if interrupted.
    best_ns_cfg.resume = "resume"

    # The best case is only five-dimensional and can become extremely
    # concentrated; the default UltraNest region sampler is less prone
    # to long, apparently frozen slice-sampling moves here.
    best_ns_cfg.use_stepsampler = False

    t0 = time.time()

    nested_best = wd0806.run_nested_validation(
        cfg,
        ns_cfg=best_ns_cfg,
        case_bodies={
            "best_case": validation_bodies["best_case"],
        },
        output_dir="result_wd0806_v16_nested_best",
        progress=True,
    )

    print(f"Nested best-case runtime: {time.time() - t0:.2f} seconds")
    print(nested_best["report"])
    display(nested_best["comparison"])

    # fig, ax = wd0806.plot_nested_validation(
    #     nested_best,
    #     case="best_case",
    # )
    # plt.show()

In [ ]:
# Worst-case evidence diagnostic: kept separate so it is not lost if another case fails.
RUN_NESTED_WORST = True
nested_worst = None

if RUN_NESTED_WORST:
    t0 = time.time()
    nested_worst = wd0806.run_nested_validation(
        cfg,
        ns_cfg=nested_config(max_ncalls=250_000),
        case_bodies={"worst_case": validation_bodies["worst_case"]},
        output_dir="result_wd0806_v16_nested_worst",
        progress=True,
    )
    print(f"Nested worst-case runtime: {time.time() - t0:.2f} seconds")
    print(nested_worst["report"])
    display(nested_worst["comparison"])

    fig, ax = wd0806.plot_nested_validation(nested_worst, case="worst_case")
    plt.show()


In [ ]:
# Worst-case M1 corner plot
try:
    fig_corner = wd0806.plot_nested_corner(
        nested_worst,
        case="worst_case",
        min_spread=1e-20,
    )

    fig_corner.suptitle(
        "Worst-case M1 posterior",
        fontsize=15,
        y=1.01,
    )

    for ax in fig_corner.axes:
        ax.tick_params(
            axis="both",
            which="both",
            labelsize=9,
        )
        ax.xaxis.label.set_size(14)
        ax.yaxis.label.set_size(14)
        ax.title.set_size(15)

    fig_corner.tight_layout()
    plt.show()

except ValueError as exc:
    print(exc)
    print(
        "The worst-case run completed, but its retained M1 posterior "
        "does not have enough dynamic range for a meaningful corner plot."
    )

In [ ]:
# Intermediate posterior-shape diagnostic: intentionally has no max_ncalls cap.
RUN_NESTED_INTERMEDIATE = True
nested_intermediate = None

if RUN_NESTED_INTERMEDIATE:
    t0 = time.time()
    nested_intermediate = wd0806.run_nested_validation(
        cfg,
        ns_cfg=nested_config(max_ncalls=None),
        case_bodies={"intermediate_case": validation_bodies["intermediate_case"]},
        output_dir="result_wd0806_v16_nested_intermediate",
        progress=True,
    )
    print(f"Nested intermediate-case runtime: {time.time() - t0:.2f} seconds")
    print(nested_intermediate["report"])
    display(nested_intermediate["comparison"])

    fig, ax = wd0806.plot_nested_validation(
        nested_intermediate,
        case="intermediate_case",
    )
    plt.show()


In [ ]:
# The uncapped intermediate run is the intended corner-plot diagnostic.
if nested_intermediate is not None:
    try:
        fig = wd0806.plot_nested_corner(
            nested_intermediate,
            case="intermediate_case",
        )
        plt.show()
    except ValueError as exc:
        print(exc)
        print(
            "The run completed, but the retained M1 posterior still lacks enough "
            "spread for a meaningful corner plot. Do not use a degenerate corner "
            "plot as evidence of detectability."
        )
else:
    print("Run the intermediate nested-validation cell before creating its corner plot.")


### Nested-validation interpretation

The current nested model uses the full configured radius and separation ranges rather than priors narrowed around the injected values. Nevertheless, Bayesian evidence is prior-sensitive. Narrowing the priors around an injected signal could artificially increase the evidence for \(M1\), so even a nominal worst-case injection might appear to favour \(M1\) strongly.

For that reason, the nested-sampling Bayes factors are reported only as diagnostic model comparisons. The principal detectability statistic remains the result from the injection--recovery grid and its matched no-object false-positive grid, not the nested-sampling result.


### Bayes-factor interpretation using Jeffreys' scale

For the nested-sampling comparison,

$$
\ln K = \ln Z_{M1} - \ln Z_{M0},
$$

where \(M0\) is the no-object model and \(M1\) is the body-transit model. Positive \(\ln K\) favours \(M1\); negative \(\ln K\) favours \(M0\).

| \(K = Z_{M1}/Z_{M0}\) | \(\log_{10}K\) | \(\ln K\) | Interpretation for \(M1\) over \(M0\) |
|---:|---:|---:|---|
| \(1 \text{ to } 10^{1/2}\) | 0 to 0.5 | 0 to 1.15 | Barely worth mentioning |
| \(10^{1/2} \text{ to } 10\) | 0.5 to 1.0 | 1.15 to 2.30 | Substantial |
| \(10 \text{ to } 10^{3/2}\) | 1.0 to 1.5 | 2.30 to 3.45 | Strong |
| \(10^{3/2} \text{ to } 10^2\) | 1.5 to 2.0 | 3.45 to 4.61 | Very strong |
| \(>10^2\) | \(>2.0\) | \(>4.61\) | Decisive |

These labels are interpretive guidelines, not calibrated detection thresholds.
